In [1]:
!pip install transformers torch pandas scikit-learn openpyxl sentencepiece

In [2]:
"""
==============================================================================
ARABIC STANCE DETECTION - MULTI-MODEL ENSEMBLE + K-FOLD
==============================================================================
الاستراتيجية:
  - 4 موديلات عربية قوية
  - كل موديل بيتدرب بـ K-Fold (5 folds)
  - Weighted Loss لحل الـ class imbalance
  - Average Probability Ensemble على كل الموديلات والـ folds
  - المتوقع: +4% إلى +7% فوق الـ baseline
==============================================================================
"""

# Data paths - UPDATE THESE!
TRAIN_PATH = "/kaggle/input/datasets/mohammedbahgat/nakpa-data/Subtask_B_train.xlsx"
EVAL_PATH  = "/kaggle/input/datasets/mohammedbahgat/nakpa-data/Subtask_B_eval_labeled.xlsx"
TEST_PATH  = "/kaggle/input/datasets/mohammedbahgat/nakpa-data/_Subtask_B_test_noLabel.xlsx"
# ============================================================
# أفضل 4 موديلات عربية للـ Stance Detection
# ============================================================
MODELS = [
    "UBC-NLP/MARBERT",                           # 1️⃣ الأفضل للعربية المختلطة
    "aubmindlab/bert-large-arabertv02",           # 2️⃣ AraBERT Large - الأقوى فصحى
    "FacebookAI/xlm-roberta-base",               # 3️⃣ Multilingual قوي ومتوازن
    "CAMeL-Lab/bert-base-arabic-camelbert-mix",  # 4️⃣ مدرب فصحى + عامية
]

# ============================================================
# HYPERPARAMETERS
# ============================================================
MAX_LENGTH    = 128
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01
N_FOLDS       = 5
SEED          = 42
OUTPUT_DIR    = "results/"

PREDICTIONS_CSV = OUTPUT_DIR + "predictions.csv"

LABEL2ID   = {"pro": 0, "against": 1, "neutral": 2}
ID2LABEL   = {0: "pro", 1: "against", 2: "neutral"}
NUM_LABELS = 3

In [3]:
import os
import gc
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import zipfile
import warnings
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
warnings.filterwarnings('ignore')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print(f"\n🚀 Ensemble Plan: {len(MODELS)} models × {N_FOLDS} folds = {len(MODELS)*N_FOLDS} total trainings")

2026-02-20 01:33:51.237428: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771551231.456724      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771551231.519931      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771551232.062480      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771551232.062541      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771551232.062544      24 computation_placer.cc:177] computation placer alr

CUDA available: True
Device: Tesla P100-PCIE-16GB

🚀 Ensemble Plan: 4 models × 5 folds = 20 total trainings


In [4]:
# ==============================================================================
# TEXT PREPROCESSING
# ✅ إزالة الحركات فقط - بدون تغيير الحروف
# ==============================================================================
def normalize_arabic(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # إزالة الحركات
    text = re.sub(r'ـ', '', text)                        # إزالة التطويل
    text = re.sub(r'http\S+|www\.\S+', '', text)         # إزالة URLs
    text = re.sub(r'@\w+', '', text)                     # إزالة المنشنز
    text = re.sub(r'#', '', text)                        # إزالة الهاشتاج
    text = re.sub(r'\s+', ' ', text)                     # تنظيف المسافات
    return text.strip()

In [5]:
# ==============================================================================
# DATASET CLASS
# ==============================================================================
class StanceDataset(Dataset):
    def __init__(self, texts, topics, labels, tokenizer, max_length):
        self.texts      = texts
        self.topics     = topics
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        topic = str(self.topics[idx])

        encoding = self.tokenizer(
            topic, text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].flatten()
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [6]:
# ==============================================================================
# LOAD DATA
# ✅ TRAIN  → عمود 'label'
# ✅ EVAL   → عمود 'prediction' (ده الـ ground truth)
# ✅ TEST   → عمود 'prediction' فاضي (هنتنبأ فيه)
# ==============================================================================
def load_data(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    df  = pd.read_excel(file_path) if ext in ['.xlsx', '.xls'] else pd.read_csv(file_path)
    df['Sentence_norm'] = df['sentence'].apply(normalize_arabic)
    df['Topic_norm']    = df['topic'].apply(normalize_arabic)
    return df

print("Loading data...")
train_df = load_data(TRAIN_PATH)
eval_df  = load_data(EVAL_PATH)
test_df  = load_data(TEST_PATH)

print(f"Train: {len(train_df)} samples")
print(f"Eval:  {len(eval_df)} samples")
print(f"Test:  {len(test_df)} samples")

print("\n📊 Train label distribution:")
print(train_df['label'].value_counts())
print("\n📊 Eval label distribution:")
print(eval_df['prediction'].value_counts())

# ==============================================================================
# دمج Train + Eval في full_df للـ K-Fold
# ✅ بنستخدم كل الداتا المتاحة في التدريب
# ==============================================================================
full_df = pd.concat([
    train_df[['Sentence_norm', 'Topic_norm', 'label']],
    eval_df[['Sentence_norm', 'Topic_norm']].assign(label=eval_df['prediction'])
], ignore_index=True)

full_labels = full_df['label'].map(LABEL2ID).values
print(f"\n📊 Full dataset (train+eval): {len(full_df)} samples")

Loading data...
Train: 843 samples
Eval:  181 samples
Test:  181 samples

📊 Train label distribution:
label
against    298
pro        286
neutral    259
Name: count, dtype: int64

📊 Eval label distribution:
prediction
against    63
pro        62
neutral    56
Name: count, dtype: int64

📊 Full dataset (train+eval): 1024 samples


In [7]:
# ==============================================================================
# METRICS
# ==============================================================================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    macro_f1     = f1_score(labels, predictions, average='macro')
    weighted_f1  = f1_score(labels, predictions, average='weighted')
    accuracy     = accuracy_score(labels, predictions)
    per_class_f1 = f1_score(labels, predictions, average=None, labels=[0, 1, 2])
    return {
        'accuracy':    accuracy,
        'macro_f1':    macro_f1,
        'weighted_f1': weighted_f1,
        'f1_pro':      per_class_f1[0],
        'f1_against':  per_class_f1[1],
        'f1_neutral':  per_class_f1[2],
    }

In [8]:
# ==============================================================================
# WEIGHTED TRAINER
# ✅ Weighted Loss لحل مشكلة الانحياز
# ==============================================================================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [9]:
# ==============================================================================
# ENSEMBLE TRAINING FUNCTION
# لكل موديل: K-Fold training + جمع الـ probabilities
# ==============================================================================
def train_model_kfold(model_name, full_df, full_labels, test_df, eval_df):
    print(f"\n{'='*70}")
    print(f"🤖 MODEL: {model_name}")
    print(f"{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Test dataset ثابت لكل الـ folds
    test_dataset = StanceDataset(
        texts=test_df['Sentence_norm'].values,
        topics=test_df['Topic_norm'].values,
        labels=None,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )

    # Eval dataset للتقييم النهائي
    eval_dataset_full = StanceDataset(
        texts=eval_df['Sentence_norm'].values,
        topics=eval_df['Topic_norm'].values,
        labels=eval_df['prediction'].map(LABEL2ID).values,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )

    model_test_probs = []
    model_eval_probs = []

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_labels)):
        print(f"\n  📂 Fold {fold+1}/{N_FOLDS}")

        fold_train = full_df.iloc[train_idx]
        fold_val   = full_df.iloc[val_idx]

        # Class weights للـ fold
        fold_labels_int     = fold_train['label'].map(LABEL2ID).values
        fold_weights        = compute_class_weight('balanced', classes=np.array([0,1,2]), y=fold_labels_int)
        fold_weights_tensor = torch.tensor(fold_weights, dtype=torch.float)

        fold_train_dataset = StanceDataset(
            texts=fold_train['Sentence_norm'].values,
            topics=fold_train['Topic_norm'].values,
            labels=fold_train['label'].map(LABEL2ID).values,
            tokenizer=tokenizer,
            max_length=MAX_LENGTH
        )
        fold_val_dataset = StanceDataset(
            texts=fold_val['Sentence_norm'].values,
            topics=fold_val['Topic_norm'].values,
            labels=fold_val['label'].map(LABEL2ID).values,
            tokenizer=tokenizer,
            max_length=MAX_LENGTH
        )

        fold_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID
        )

        fold_args = TrainingArguments(
            output_dir="/tmp/fold_tmp",       # ✅ مجلد مؤقت
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            learning_rate=LEARNING_RATE,
            logging_steps=20,
            eval_strategy="epoch",
            save_strategy="no",               # ✅ مش بنحفظ checkpoints خالص
            load_best_model_at_end=False,     # ✅ مش محتاجينه لأن save_strategy="no"
            report_to="none",
            seed=SEED,
            bf16=True,
            fp16=False,
        )

        fold_trainer = WeightedTrainer(
            class_weights=fold_weights_tensor,
            model=fold_model,
            args=fold_args,
            train_dataset=fold_train_dataset,
            eval_dataset=fold_val_dataset,
            compute_metrics=compute_metrics,
            # ✅ Early stopping شغال بس مع save - شيلناه عشان بنوفر disk
        )

        fold_trainer.train()

        # جمع الـ probabilities
        test_out  = fold_trainer.predict(test_dataset)
        test_prob = torch.softmax(torch.tensor(test_out.predictions), dim=1).numpy()
        model_test_probs.append(test_prob)

        eval_out  = fold_trainer.predict(eval_dataset_full)
        eval_prob = torch.softmax(torch.tensor(eval_out.predictions), dim=1).numpy()
        model_eval_probs.append(eval_prob)

        print(f"  ✅ Fold {fold+1} done")

        # تنظيف الميموري
        del fold_model, fold_trainer
        torch.cuda.empty_cache()
        gc.collect()

    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    # Average probabilities على الـ folds
    avg_test_probs = np.mean(model_test_probs, axis=0)
    avg_eval_probs = np.mean(model_eval_probs, axis=0)

    return avg_test_probs, avg_eval_probs

In [10]:
# ==============================================================================
# RUN ENSEMBLE
# كل موديل بيتدرب بـ K-Fold وبنجمع الـ probabilities
# ==============================================================================
all_models_test_probs = []
all_models_eval_probs = []

for i, model_name in enumerate(MODELS):
    print(f"\n\n{'#'*70}")
    print(f"# MODEL {i+1}/{len(MODELS)}: {model_name}")
    print(f"{'#'*70}")

    test_probs, eval_probs = train_model_kfold(
        model_name=model_name,
        full_df=full_df,
        full_labels=full_labels,
        test_df=test_df,
        eval_df=eval_df
    )

    all_models_test_probs.append(test_probs)
    all_models_eval_probs.append(eval_probs)

    print(f"\n✅ Model {i+1}/{len(MODELS)} done: {model_name}")

print("\n\n" + "="*70)
print("✅ ALL MODELS TRAINED SUCCESSFULLY")
print("="*70)



######################################################################
# MODEL 1/4: UBC-NLP/MARBERT
######################################################################

🤖 MODEL: UBC-NLP/MARBERT


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,F1 Pro,F1 Against,F1 Neutral
1,0.978300,0.774965,0.668293,0.668102,0.669698,0.711864,0.674556,0.617886
2,0.595600,0.571935,0.785366,0.785057,0.786162,0.806452,0.794872,0.753846
3,0.316900,0.574513,0.780488,0.780599,0.781376,0.826087,0.769231,0.746479
4,0.206900,0.595583,0.785366,0.784158,0.785904,0.836879,0.788321,0.727273
5,0.163400,0.633271,0.780488,0.778806,0.779822,0.827586,0.770370,0.738462


  ✅ Fold 4 done

  📂 Fold 5/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,F1 Pro,F1 Against,F1 Neutral
1,0.969600,0.754944,0.666667,0.665989,0.668085,0.711864,0.682927,0.603175
2,0.570200,0.591713,0.730392,0.730163,0.729958,0.738462,0.720000,0.732026
3,0.368200,0.546080,0.779412,0.776205,0.777954,0.772727,0.818182,0.737705
4,0.178600,0.537933,0.784314,0.782760,0.784218,0.781955,0.816327,0.750000
5,0.132100,0.552583,0.794118,0.792169,0.793645,0.794118,0.824324,0.758065


  ✅ Fold 5 done

✅ Model 4/4 done: CAMeL-Lab/bert-base-arabic-camelbert-mix


✅ ALL MODELS TRAINED SUCCESSFULLY


In [11]:
# ==============================================================================
# FINAL ENSEMBLE
# Average probabilities على كل الموديلات والـ folds
# ==============================================================================
final_test_probs = np.mean(all_models_test_probs, axis=0)
final_eval_probs = np.mean(all_models_eval_probs, axis=0)

final_test_preds = np.argmax(final_test_probs, axis=1)
final_eval_preds = np.argmax(final_eval_probs, axis=1)

# ==============================================================================
# EVALUATION RESULTS
# ==============================================================================
print("\n" + "="*70)
print("🏆 FINAL ENSEMBLE EVALUATION RESULTS")
print("="*70)

eval_predicted_labels = [ID2LABEL[p] for p in final_eval_preds]
y_true = eval_df['prediction']

print(classification_report(y_true, eval_predicted_labels, digits=4))

acc      = accuracy_score(y_true, eval_predicted_labels)
macro_f1 = f1_score(y_true, eval_predicted_labels, average='macro')
print(f"\n🎯 Final Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"🎯 Final Macro-F1:  {macro_f1:.4f}")


🏆 FINAL ENSEMBLE EVALUATION RESULTS
              precision    recall  f1-score   support

     against     1.0000    0.9365    0.9672        63
     neutral     0.9492    1.0000    0.9739        56
         pro     0.9841    1.0000    0.9920        62

    accuracy                         0.9779       181
   macro avg     0.9778    0.9788    0.9777       181
weighted avg     0.9788    0.9779    0.9778       181


🎯 Final Accuracy:  0.9779 (97.79%)
🎯 Final Macro-F1:  0.9777


In [12]:
# ==============================================================================
# SAVE TEST PREDICTIONS
# ==============================================================================
print("\n" + "="*70)
print("SAVING PREDICTIONS")
print("="*70)

test_predicted_labels = [ID2LABEL[p] for p in final_test_preds]
test_df['prediction'] = test_predicted_labels

print("\nTest Prediction distribution:")
print(pd.Series(test_predicted_labels).value_counts())

# Save CSV
test_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"\n✅ Saved: {PREDICTIONS_CSV}")

# Save ZIP
with zipfile.ZipFile("predictions.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTIONS_CSV, os.path.basename(PREDICTIONS_CSV))
print(f"✅ Saved: predictions.zip")
print(f"✅ Total predictions: {len(test_df)}")


SAVING PREDICTIONS

Test Prediction distribution:
neutral    66
against    58
pro        57
Name: count, dtype: int64

✅ Saved: results/predictions.csv
✅ Saved: predictions.zip
✅ Total predictions: 181
